In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

# -----------------------------
# Load data
# -----------------------------
csv_files = list(Path(".").glob("Teen_Mental_Health_Dataset*.csv"))
if not csv_files:
    raise FileNotFoundError("Teen Mental Health CSV not found in the current folder.")

df = pd.read_csv(csv_files[0])

# Standardize column types
text_cols = ["gender", "platform_usage", "social_interaction_level"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string")

numeric_like_cols = [
    "age",
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "depression_label",
]
for col in numeric_like_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Q1: First 10 rows")
display(df.head(10))

numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("\nQ1: Numerical variables")
print(numeric_cols)

print("\nQ1: Categorical variables")
print(categorical_cols)

print("\nQ1: Data types")
print(df.dtypes)

print("\nQ1: Type conversion note")
print("Treat depression_label as a binary target/category for interpretation, even though it is stored numerically.")
print("Gender, platform_usage, and social_interaction_level are categorical.")
print("If needed for modeling, social_interaction_level can be ordinal-encoded later.")

# -----------------------------
# Q2: Descriptive statistics
# -----------------------------
print("\nQ2a: Descriptive statistics for all numerical variables")
desc = df[numeric_cols].describe().T
desc["std"] = df[numeric_cols].std(numeric_only=True)
display(desc)

highest_variation_var = df[numeric_cols].std(numeric_only=True).idxmax()
print(f"Variable with highest variation by standard deviation: {highest_variation_var}")

print("\nQ2b: Mean vs median for daily_social_media_hours and sleep_hours")
compare_cols = ["daily_social_media_hours", "sleep_hours"]
summary_compare = pd.DataFrame({
    "mean": df[compare_cols].mean(),
    "median": df[compare_cols].median(),
    "difference(mean-median)": df[compare_cols].mean() - df[compare_cols].median(),
})
display(summary_compare)

for col in compare_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    diff = mean_val - median_val
    if pd.isna(diff):
        print(f"{col}: not enough data to interpret.")
    elif diff > 0:
        print(f"{col}: mean > median, suggesting right-skew / positive skew.")
    elif diff < 0:
        print(f"{col}: mean < median, suggesting left-skew / negative skew.")
    else:
        print(f"{col}: mean and median are equal, suggesting a more symmetric distribution.")

# -----------------------------
# Q3: Missing values and imputation guidance
# -----------------------------
print("\nQ3a: Missing values by column")
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
missing = missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)
display(missing)

print("\nQ3b: Suitable imputation for daily_social_media_hours, sleep_hours, screen_time_before_sleep")
print("Use median imputation first, because these are numeric behavioral variables and median is robust to outliers.")
print("If you want a stronger method, use KNN imputation or regression-based imputation with related features.")

print("\nQ3c: Suitable imputation for stress_level, anxiety_level, addiction_level")
print("If they are treated as numeric severity scores, median imputation is safe.")
print("If they are interpreted as ordered levels, use ordinal-aware imputation or model-based imputation.")
print("Do not use mean if the distribution is strongly skewed or contains outliers.")

print("\nQ3d: MAR vs MNAR explanation")
print("MAR: anxiety_level missingness may depend on observed variables such as sleep_hours, stress_level, or age.")
print("MNAR: students with very high anxiety may be less likely to answer the anxiety question itself, so missingness depends on the unseen anxiety value.")

# -----------------------------
# Q4: Visualizations
# -----------------------------
plot_df = df.copy()
plot_df["gender"] = plot_df["gender"].fillna("Missing").astype("string")
plot_df["platform_usage"] = plot_df["platform_usage"].fillna("Missing").astype("string")
plot_df["social_interaction_level"] = plot_df["social_interaction_level"].fillna("Missing").astype("string")

label_map = {0: "No Depression", 1: "Depression"}
plot_df["depression_label_text"] = plot_df["depression_label"].map(label_map).fillna("Missing")

print("\nQ4a: Distribution plots")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=plot_df, x="gender", ax=axes[0], order=plot_df["gender"].value_counts().index)
axes[0].set_title("Gender Distribution")
axes[0].tick_params(axis="x", rotation=20)

sns.countplot(data=plot_df, x="platform_usage", ax=axes[1], order=plot_df["platform_usage"].value_counts().index)
axes[1].set_title("Platform Usage Distribution")
axes[1].tick_params(axis="x", rotation=20)

sns.countplot(data=plot_df, x="depression_label_text", ax=axes[2], order=plot_df["depression_label_text"].value_counts().index)
axes[2].set_title("Depression Label Distribution")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print("\nQ4b: Boxplots grouped by depression label")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["stress_level", "anxiety_level", "addiction_level"]):
    sns.boxplot(data=plot_df, x="depression_label_text", y=col, ax=ax)
    ax.set_title(f"{col} by Depression Label")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print("\nQ4c: Scatter plots")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(data=plot_df, x="daily_social_media_hours", y="stress_level", hue="depression_label_text", ax=axes[0], alpha=0.7)
axes[0].set_title("Daily Social Media Hours vs Stress Level")

sns.scatterplot(data=plot_df, x="sleep_hours", y="anxiety_level", hue="depression_label_text", ax=axes[1], alpha=0.7)
axes[1].set_title("Sleep Hours vs Anxiety Level")

plt.tight_layout()
plt.show()

print("\nQ4d: Quick chart interpretation helper")
print("1. Countplots show whether the sample is balanced across gender, platform usage, and depression status.")
print("2. Boxplots show whether stress/anxiety/addiction tend to be higher in the depression group.")
print("3. Scatterplots show whether more social media time is linked to stress and whether less sleep is linked to anxiety.")

# -----------------------------
# Q5: Correlation analysis
# -----------------------------
corr_cols = [col for col in numeric_cols if col != "depression_label"]
corr = df[corr_cols].corr()

print("\nQ5: Correlation matrix")
display(corr)

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

def corr_strength(r):
    a = abs(r)
    if a < 0.2:
        return "very weak"
    elif a < 0.4:
        return "weak"
    elif a < 0.6:
        return "moderate"
    elif a < 0.8:
        return "strong"
    return "very strong"

focus_pairs = [
    ("daily_social_media_hours", "sleep_hours"),
    ("daily_social_media_hours", "stress_level"),
    ("daily_social_media_hours", "anxiety_level"),
    ("daily_social_media_hours", "addiction_level"),
    ("sleep_hours", "stress_level"),
    ("sleep_hours", "anxiety_level"),
    ("stress_level", "anxiety_level"),
    ("stress_level", "addiction_level"),
    ("anxiety_level", "addiction_level"),
    ("academic_performance", "stress_level"),
    ("academic_performance", "anxiety_level"),
]

print("\nQ5: Targeted correlation interpretation")
for a, b in focus_pairs:
    if a in corr.columns and b in corr.columns:
        r = corr.loc[a, b]
        direction = "positive" if r > 0 else "negative"
        print(f"{a} vs {b}: r = {r:.2f} -> {corr_strength(r)} {direction} correlation")

print("\nQ5: Why categorical variables should not be directly included")
print("Correlation requires numeric values with meaningful distances.")
print("Categorical variables like gender or platform_usage must be encoded carefully first, otherwise the numeric codes would create fake ordering and misleading correlations.")

# -----------------------------
# Q6: Outlier detection using IQR
# -----------------------------
outlier_cols = ["daily_social_media_hours", "sleep_hours", "stress_level", "anxiety_level"]
outlier_results = []

for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_results.append({
        "variable": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": len(outliers),
    })

outlier_table = pd.DataFrame(outlier_results)
print("\nQ6a: IQR-based outlier summary")
display(outlier_table)

for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"\n{col}: {len(outliers)} outliers")
    if len(outliers) > 0:
        display(outliers[[col]].head(10))

print("\nQ6b: Interpretation guide")
print("Extreme social media values may be real behavior if they align with other high-risk patterns.")
print("If they look impossible or inconsistent with the rest of the record, they may be data errors.")
print("Check whether outliers also appear in stress, anxiety, addiction, sleep, and screen time before deciding to remove them.")

# -----------------------------
# Q7: Deeper analysis by depression category
# -----------------------------
print("\nQ7a-Q7d: Group comparison by depression label")
analysis_df = df.copy()
analysis_df["depression_label_text"] = analysis_df["depression_label"].map(label_map).fillna("Missing")

# Ordinal helper for social interaction level
social_map = {"low": 1, "medium": 2, "high": 3}
analysis_df["social_interaction_level_ord"] = (
    analysis_df["social_interaction_level"].astype("string").str.lower().map(social_map)
)

group_cols = [
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "academic_performance",
    "physical_activity",
    "social_interaction_level_ord",
]
group_summary = analysis_df.groupby("depression_label_text")[group_cols].agg(["mean", "median"])
display(group_summary)

print("\nQ7a: Relationship summary")
print("Higher social media usage often aligns with higher stress, anxiety, and addiction if the group means and scatter trends support it.")

print("\nQ7b: Sleep and screen time summary")
sleep_screen_summary = analysis_df.groupby("depression_label_text")[["sleep_hours", "screen_time_before_sleep"]].agg(["mean", "median"])
display(sleep_screen_summary)
print("Lower sleep and higher screen time before sleep generally indicate worse mental health patterns if the depressed group shows those trends.")

print("\nQ7c: Most significant contributors to depression")
print("Inspect the boxplots, correlations, and group means for stress_level, anxiety_level, addiction_level, sleep_hours, and screen_time_before_sleep.")
print("Those variables usually provide the clearest signal in this kind of dataset.")

print("\nQ7d: Category comparison plots")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.boxplot(data=analysis_df, x="depression_label_text", y="academic_performance", ax=axes[0, 0])
axes[0, 0].set_title("Academic Performance by Depression Label")
sns.boxplot(data=analysis_df, x="depression_label_text", y="physical_activity", ax=axes[0, 1])
axes[0, 1].set_title("Physical Activity by Depression Label")
sns.boxplot(data=analysis_df, x="depression_label_text", y="social_interaction_level_ord", ax=axes[1, 0])
axes[1, 0].set_title("Social Interaction Level by Depression Label")
sns.boxplot(data=analysis_df, x="depression_label_text", y="sleep_hours", ax=axes[1, 1])
axes[1, 1].set_title("Sleep Hours by Depression Label")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()